# Train and evaluate the fraud model

Use only feature rows marked `KEEP` in `feature_selection_report.csv`. Train a picklable scikit-learn pipeline, evaluate on validation and held-out test data, and save structured metrics and model artifacts under `model_development/artifacts`.

In [ ]:
import json
import os
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
pd.set_option("display.max_columns", None)
import xgboost as xgb
from pyspark.sql import SparkSession, functions as F
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, classification_report, confusion_matrix, f1_score,
    log_loss, matthews_corrcoef, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "data" / "processed" / "transactions_parquet").is_dir()
     and (candidate / "model_development" / "feature_selection_report" / "feature_selection_report.csv").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate processed transactions and the feature selection report from the notebook directory.")

processed_data_path = project_root / "data" / "processed" / "transactions_parquet"
feature_report_path = project_root / "model_development" / "feature_selection_report" / "feature_selection_report.csv"
feature_report = pd.read_csv(feature_report_path)
required_report_columns = {"COLUMN_NAME", "FINAL_FLAG", "ROLE"}
missing_report_columns = sorted(required_report_columns - set(feature_report.columns))
if missing_report_columns:
    raise ValueError(f"Feature report is missing required columns: {missing_report_columns}")

keep_mask = (
    feature_report["FINAL_FLAG"].astype("string").str.strip().str.upper().eq("KEEP")
    & feature_report["ROLE"].astype("string").str.strip().str.upper().eq("FEATURE")
)
selected_features = feature_report.loc[keep_mask, "COLUMN_NAME"].dropna().astype(str).drop_duplicates().tolist()
label_column = "is_fraud"
if not selected_features:
    raise ValueError("The feature report contains no rows with FINAL_FLAG=KEEP and ROLE=FEATURE.")
if label_column in selected_features:
    raise ValueError("The target is_fraud must not be used as a model feature.")

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("FraudXGBoostModelTraining")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
source_df = spark.read.parquet(str(processed_data_path))
required_columns = ["trans_num", label_column, *selected_features]
missing_columns = sorted(set(required_columns) - set(source_df.columns))
if missing_columns:
    raise ValueError(f"Processed Parquet data is missing report-selected columns: {missing_columns}")

training_sdf = (
    source_df.select(*required_columns)
    .filter(F.col(label_column).isin(0, 1) & F.col("trans_num").isNotNull())
    .dropDuplicates(["trans_num"])
)
training_pdf = training_sdf.toPandas()
if training_pdf.empty:
    raise ValueError("No labeled transactions were found in the processed Parquet data.")

labels = pd.to_numeric(training_pdf[label_column], errors="raise").astype(np.int8)
if set(labels.unique()) != {0, 1}:
    raise ValueError("Training requires both legitimate (0) and fraud (1) examples.")
features = training_pdf[selected_features].copy()
numeric_features = [column for column in selected_features if pd.api.types.is_numeric_dtype(features[column])]
categorical_features = [column for column in selected_features if column not in numeric_features]
for column in categorical_features:
    features[column] = features[column].astype("string").fillna("Unknown").astype(str)

print(f"Loaded {len(training_pdf):,} unique labeled transactions")
print(f"Using {len(selected_features)} KEEP features from {feature_report_path}")
print("Selected features:", selected_features)
print("Label counts (0 = legitimate, 1 = fraud):")
print(labels.value_counts().sort_index())
print(f"{labels.value_counts(normalize=True) * 100}")

Loaded 230,783 unique labeled transactions
Using 17 KEEP features from c:\Users\admin\Desktop\data_science_projects\real_time_fraud_transaction_detection\model_development\feature_selection_report\feature_selection_report.csv
Selected features: ['amt', 'amt_log1p', 'category', 'cc_num', 'city', 'city_pop', 'dob', 'first', 'job', 'last', 'merch_zipcode', 'merchant', 'state', 'street', 'transaction_hour', 'unix_time', 'zip']
Label counts (0 = legitimate, 1 = fraud):
is_fraud
0    229015
1      1768
Name: count, dtype: int64


In [2]:
# Stratified 70/15/15 train, validation, and test split.
X_train_validation, X_test, y_train_validation, y_test = train_test_split(
    features, labels, test_size=0.15, random_state=42, stratify=labels
)
X_train, X_validation, y_train, y_validation = train_test_split(
    X_train_validation, y_train_validation, test_size=(0.15 / 0.85),
    random_state=42, stratify=y_train_validation
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
classifier = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric=["auc", "aucpr"],
    tree_method="hist",
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=negative_count / positive_count,
    random_state=42,
    n_jobs=-1,
)
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", classifier),
])
model_pipeline.fit(X_train, y_train)
print(f"Training rows: {len(X_train):,}; validation rows: {len(X_validation):,}; test rows: {len(X_test):,}")
print(f"Fraud class weight (scale_pos_weight): {negative_count / positive_count:.2f}")

Training rows: 161,547; validation rows: 34,618; test rows: 34,618
Fraud class weight (scale_pos_weight): 129.49


In [3]:
# Report threshold-based and probability-based metrics for both validation and test splits.
prediction_threshold = 0.5
evaluation_results = {}
confusion_matrices = {}
for split_name, X_split, y_split in [
    ("validation", X_validation, y_validation),
    ("test", X_test, y_test),
]:
    probabilities = model_pipeline.predict_proba(X_split)[:, 1]
    predictions = (probabilities >= prediction_threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(y_split, predictions, labels=[0, 1]).ravel()
    auc = float(roc_auc_score(y_split, probabilities))
    metric_values = {
        "n_samples": int(len(y_split)),
        "prediction_threshold": prediction_threshold,
        "accuracy": float(accuracy_score(y_split, predictions)),
        "balanced_accuracy": float(balanced_accuracy_score(y_split, predictions)),
        "precision": float(precision_score(y_split, predictions, zero_division=0)),
        "recall": float(recall_score(y_split, predictions, zero_division=0)),
        "f1": float(f1_score(y_split, predictions, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if (tn + fp) else 0.0,
        "negative_predictive_value": float(tn / (tn + fn)) if (tn + fn) else 0.0,
        "roc_auc": auc,
        "gini": float(2 * auc - 1),
        "average_precision": float(average_precision_score(y_split, probabilities)),
        "log_loss": float(log_loss(y_split, probabilities, labels=[0, 1])),
        "brier_score": float(brier_score_loss(y_split, probabilities)),
        "matthews_corrcoef": float(matthews_corrcoef(y_split, predictions)),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }
    evaluation_results[split_name] = {
        "metrics": metric_values,
        "classification_report": classification_report(
            y_split, predictions, labels=[0, 1],
            target_names=["legitimate", "fraud"], output_dict=True, zero_division=0
        ),
    }
    confusion_matrices[split_name] = pd.DataFrame(
        [[int(tn), int(fp)], [int(fn), int(tp)]],
        index=["actual_legitimate", "actual_fraud"],
        columns=["predicted_legitimate", "predicted_fraud"],
    )

for split_name in ("validation", "test"):
    metrics = evaluation_results[split_name]["metrics"]
    print(f"{split_name.title()} metrics at threshold {prediction_threshold:.2f}:")
    print(f"  ROC-AUC: {metrics['roc_auc']:.6f}; Gini: {metrics['gini']:.6f}; PR-AUC: {metrics['average_precision']:.6f}")
    print(f"  Accuracy: {metrics['accuracy']:.6f}; Precision: {metrics['precision']:.6f}; Recall: {metrics['recall']:.6f}; F1: {metrics['f1']:.6f}")
    print(f"  Balanced accuracy: {metrics['balanced_accuracy']:.6f}; Specificity: {metrics['specificity']:.6f}; MCC: {metrics['matthews_corrcoef']:.6f}")
    print("  Confusion matrix [ [TN, FP], [FN, TP] ]:")
    print(confusion_matrices[split_name])

Validation metrics at threshold 0.50:
  ROC-AUC: 0.996550; Gini: 0.993100; PR-AUC: 0.906825
  Accuracy: 0.992056; Precision: 0.490196; Recall: 0.943396; F1: 0.645161
  Balanced accuracy: 0.967914; Specificity: 0.992432; MCC: 0.676999
  Confusion matrix [ [TN, FP], [FN, TP] ]:
                   predicted_legitimate  predicted_fraud
actual_legitimate                 34093              260
actual_fraud                         15              250
Test metrics at threshold 0.50:
  ROC-AUC: 0.996766; Gini: 0.993532; PR-AUC: 0.900475
  Accuracy: 0.991478; Precision: 0.471483; Recall: 0.935849; F1: 0.627054
  Balanced accuracy: 0.963878; Specificity: 0.991908; MCC: 0.661029
  Confusion matrix [ [TN, FP], [FN, TP] ]:
                   predicted_legitimate  predicted_fraud
actual_legitimate                 34075              278
actual_fraud                         17              248


In [4]:
# Save stable, MLflow-friendly artifacts: JSON metrics, tabular metrics, reports, and a pickled sklearn pipeline.
artifact_dir = project_root / "model_development" / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)
metrics_payload = {
    "target": label_column,
    "positive_label": 1,
    "selected_features": selected_features,
    "feature_count": len(selected_features),
    "prediction_threshold": prediction_threshold,
    "splits": evaluation_results,
}
metrics_json_path = artifact_dir / "evaluation_metrics.json"
with metrics_json_path.open("w", encoding="utf-8") as file:
    json.dump(metrics_payload, file, indent=2, allow_nan=False)

metrics_csv_path = artifact_dir / "evaluation_metrics.csv"
pd.DataFrame([
    {"split": split_name, **result["metrics"]}
    for split_name, result in evaluation_results.items()
]).to_csv(metrics_csv_path, index=False)
for split_name, matrix in confusion_matrices.items():
    matrix.to_csv(artifact_dir / f"confusion_matrix_{split_name}.csv", index=True)
with (artifact_dir / "classification_reports.json").open("w", encoding="utf-8") as file:
    json.dump(
        {name: result["classification_report"] for name, result in evaluation_results.items()},
        file, indent=2, allow_nan=False
    )
with (artifact_dir / "selected_features.json").open("w", encoding="utf-8") as file:
    json.dump(selected_features, file, indent=2)

transformed_feature_names = model_pipeline.named_steps["preprocessor"].get_feature_names_out()
feature_importance = pd.DataFrame({
    "feature": transformed_feature_names,
    "importance": model_pipeline.named_steps["classifier"].feature_importances_,
}).sort_values("importance", ascending=False)
feature_importance.to_csv(artifact_dir / "feature_importance.csv", index=False)

model_pkl_path = artifact_dir / "fraud_model.pkl"
with model_pkl_path.open("wb") as file:
    pickle.dump(model_pipeline, file, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved pickled scikit-learn pipeline: {model_pkl_path}")
print(f"Saved metrics JSON: {metrics_json_path}")
print(f"Saved metrics CSV: {metrics_csv_path}")
print(f"Saved confusion matrices, classification reports, selected feature list, and feature importances in {artifact_dir}")
print("The pickled pipeline can be loaded as a scikit-learn estimator and logged later with mlflow.sklearn.log_model.")

Saved pickled scikit-learn pipeline: c:\Users\admin\Desktop\data_science_projects\real_time_fraud_transaction_detection\model_development\artifacts\fraud_model.pkl
Saved metrics JSON: c:\Users\admin\Desktop\data_science_projects\real_time_fraud_transaction_detection\model_development\artifacts\evaluation_metrics.json
Saved metrics CSV: c:\Users\admin\Desktop\data_science_projects\real_time_fraud_transaction_detection\model_development\artifacts\evaluation_metrics.csv
Saved confusion matrices, classification reports, selected feature list, and feature importances in c:\Users\admin\Desktop\data_science_projects\real_time_fraud_transaction_detection\model_development\artifacts
The pickled pipeline can be loaded as a scikit-learn estimator and logged later with mlflow.sklearn.log_model.


## mlflow setup